In [ ]:
import fitz
import re
from collections import Counter
from langchain_core.documents import Document


In [ ]:
def production_pdf_loader(pdf_path, header_pct=0.08, footer_pct=0.08):
    pdf = fitz.open(pdf_path)
    raw_pages = []
    print(pdf.metadata)

    # Step 1 — Extract with coordinate cropping (skip cover / title page)
    for page_num, page in enumerate(pdf):
        if page_num == 0:
            continue
        h = page.rect.height
        w = page.rect.width
        rect = fitz.Rect(0, h * header_pct, w, h * (1 - footer_pct))
        text = page.get_text("text", clip=rect)
        raw_pages.append((page_num, text))
    pdf.close()

    # Step 2 — Remove remaining repeated lines
    all_lines = []
    for _, p in raw_pages:
        all_lines.extend([l.strip() for l in p.split("\n") if l.strip()])
    counts = Counter(all_lines)
    repeated = {l for l, c in counts.items() if c / len(raw_pages) >= 0.5}

    # Step 3 — Clean each page
    docs = []
    for page_num, text in raw_pages:
        lines = text.split("\n")
        cleaned = []
        for line in lines:
            stripped = line.strip()
            if stripped in repeated:
                continue
            if re.match(r"^\s*\d+\s*$", stripped):
                continue
            cleaned.append(line)

        # Fix word-per-line artifacts
        text = "\n".join(cleaned)
        text = re.sub(r"  +", " ", text)
        text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)

        docs.append(Document(
            page_content=text.strip(),
            metadata={**pdf.metadata, "page": page_num},
        ))

    return docs

In [ ]:
def merge_pages_by_file(documents: list) -> list:
    """Merge all pages of a PDF into one Document so splits can span page breaks."""
    groups: dict[str, list] = {}
    order: list[str] = []
    for doc in documents:
        key = doc.metadata.get("title") or "default"
        if key not in groups:
            groups[key] = []
            order.append(key)
        groups[key].append(doc)

    merged: list[Document] = []
    for key in order:
        batch = sorted(groups[key], key=lambda d: d.metadata.get("page", 0))
        text = "\n\n".join(d.page_content for d in batch)
        meta = {**batch[0].metadata, "merged_page_count": len(batch)}
        merged.append(Document(page_content=text, metadata=meta))
    return merged

In [ ]:
# Get all pdf files in the policies folder
import glob


pdf_files = glob.glob("../policies/*.pdf")

# Load all pdf files
all_pdf = []
for pdf_file in pdf_files:
    all_pdf.extend(production_pdf_loader(pdf_file))

In [ ]:
# Merge all pdf files
all_pdf_merged = merge_pages_by_file(all_pdf)

In [ ]:
all_pdf_merged

In [ ]:
from langchain_chroma import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import LocalFileStore, create_kv_docstore
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ─────────────────────────────────────────────
# STEP 3 — SPLITTERS
# Child  → small, precise  → gets embedded into Chroma
# Parent → large, complete → gets returned to the LLM
# ─────────────────────────────────────────────
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "],
)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=200,
    separators=[
        r"\n(?=\d+\.\s+[A-Z])",      # top-level section  e.g. "3. Leave Entitlements"
        r"\n(?=\d+\.\d+\.\s)",        # sub-section        e.g. "3.1.", "3.8."
        "\n\n",
        "\n",
        ". ",
    ],
    is_separator_regex=True,
)


# ─────────────────────────────────────────────
# STEP 4 — EMBEDDINGS (BGE-M3)
# ─────────────────────────────────────────────
lc_embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"trust_remote_code": True},
    encode_kwargs={"normalize_embeddings": True},
)


# ─────────────────────────────────────────────
# STEP 5 — STORES
# vectorstore  → Chroma  (child chunk embeddings, persisted to disk)
# docstore     → LocalFileStore  (parent chunks raw text, persisted to disk)
#
# ⚠️  If you change splitters, PDFs, or chunking strategy — delete both stores:
#     shutil.rmtree("bge-m3-child-store", ignore_errors=True)
#     shutil.rmtree("bge-m3-parent-store", ignore_errors=True)
# ─────────────────────────────────────────────

# Uncomment when re-embedding from scratch:
# shutil.rmtree("bge-m3-child-store", ignore_errors=True)
# shutil.rmtree("bge-m3-parent-store", ignore_errors=True)

vectorstore = Chroma(
    collection_name="bge-m3-policy-v4",
    embedding_function=lc_embeddings,
    persist_directory="bge-m3-child-store",   # child embeddings
)

# LocalFileStore persists parent docs to disk (survives restarts, unlike InMemoryStore)
fs = LocalFileStore("bge-m3-parent-store")
docstore = create_kv_docstore(fs)             # parent raw text


# ─────────────────────────────────────────────
# STEP 6 — ParentDocumentRetriever
# ─────────────────────────────────────────────
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_type="mmr",
    search_kwargs={
        "k": 4,          # child chunks to match
        "fetch_k": 24,   # MMR candidate pool
        "lambda_mult": 0.6,
    },
)

# Index documents (only needed once; re-run if you cleared the stores)
retriever.add_documents(all_pdf_merged, add_to_docstore=True)


In [ ]:
# ─────────────────────────────────────────────
# STEP 7 — QUERY
# Returns full parent chunks (3.1–3.14 together), not just the matched child
# ─────────────────────────────────────────────
results = retriever.invoke("Leave policy")
for r in results:
    # print(r.metadata)
    print(r.page_content)
    print("─" * 60)